## This program builds an AI Scout that attempts to look at a cricket player’s current form and predict whether they will fail, stabilize, or dominate in their very next match.

In [8]:
import glob
import os
import pandas as pd

### 1. The Raw Ingestion
We start with 765,058 rows of raw text. At this stage, the data has no context - it is just a massive timeline of individual balls being bowled, showing who hit the ball and how many runs they took.

In [9]:
EXTRACTION_FOLDER = "extracted_matches"

# 1. Gather all individual match CSV paths
all_match_files = glob.glob(
    os.path.join(EXTRACTION_FOLDER, "**", "*.csv"), recursive=True
)
all_match_files = [
    f
    for f in all_match_files
    if "matches.csv" not in f and "readme" not in f.lower()
]

ball_columns = [
    "record_type",
    "innings",
    "delivery_num",
    "batting_team",
    "batter",
    "non_striker",
    "bowler",
    "runs_off_bat",
    "extras",
    "wides",
    "noballs",
    "byes",
    "legbyes",
    "penalty",
    "dismissal_kind",
    "player_dismissed",
]

master_ball_data = []
print(f"Scanning {len(all_match_files)} matches directly into RAM...")

for file_path in all_match_files:
    match_id = os.path.basename(file_path).split(".")[0]
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            for line in f:
                if line.startswith("ball,"):
                    row_values = line.strip().split(",")
                    if len(row_values) < len(ball_columns):
                        row_values += [0] * (len(ball_columns) - len(row_values))
                    else:
                        row_values = row_values[: len(ball_columns)]

                    row_data = dict(zip(ball_columns, row_values))
                    row_data["match_id"] = match_id
                    master_ball_data.append(row_data)
    except:
        continue

# Create raw master DataFrame
df = pd.DataFrame(master_ball_data)
df["runs_off_bat"] = pd.to_numeric(df["runs_off_bat"], errors="coerce")
print(
    f"Initialization complete! Extracted {len(df)} rows. You can move to the next cell."
)

Scanning 3393 matches directly into RAM...
Initialization complete! Extracted 765058 rows. You can move to the next cell.


## 2. The Feature Translation
The program compresses those three-quarters of a million balls into distinct player innings and calculates three vital metrics for every batsman:
runs_scored (How many runs they made in that specific match), rolling_avg_3 (Their immediate recent form - average of their last 3 games), runs_volatility (Their consistency risk - the standard deviation of their last 5 games).

In [10]:
# Group ball tracking data into total runs scored per match per player
player_stats = (
    df.groupby(["batter", "match_id"])
    .agg(runs_scored=("runs_off_bat", "sum"), balls_faced=("record_type", "count"))
    .reset_index()
)

# Sort sequentially by player history
player_stats = player_stats.sort_values(by=["batter", "match_id"])

# 1. Target Variable: Raw runs scored in the NEXT match
player_stats["next_match_runs"] = player_stats.groupby("batter")[
    "runs_scored"
].shift(-1)

# 2. Input Feature A: Recent Form (3-match rolling average)
player_stats["rolling_avg_3"] = player_stats.groupby("batter")[
    "runs_scored"
].transform(lambda x: x.rolling(window=3, min_periods=1).mean())

# 3. Input Feature B: Volatility Index (Standard deviation of past 5 innings)
player_stats["runs_volatility"] = player_stats.groupby("batter")[
    "runs_scored"
].transform(lambda x: x.rolling(window=5, min_periods=1).std())

# Drop NaNs safely
ml_dataset = player_stats.dropna(
    subset=["next_match_runs", "runs_volatility"]
).copy()
print(
    f"Feature matrices compiled successfully! Data shape for ML modeling: {ml_dataset.shape}"
)

Feature matrices compiled successfully! Data shape for ML modeling: (46587, 7)


## 3. Setting Up the Supervised Learning Target
To train an AI, we must give it the "answer key" to learn from. The program shifts the timeline using Pandas .shift(-1) so that a player’s current row of stats sits right next to the number of runs they scored in their next chronological match. To make the problem viable for a machine learning model, it translates the chaotic raw run numbers into three distinct operational brackets: Class 0 - Early Dismissal ($<15$ runs), Class 1 - Stable Anchor ($15\text{ to }40$ runs), Class 2 - Match Winner ($>40$ runs)

In [12]:
from sklearn.model_selection import train_test_split

# Define performance target labels
def assign_performance_class(runs):
    if runs < 15:
        return 0  # Early Dismissal
    elif runs <= 40:
        return 1  # Anchor / Stable Contribution
    else:
        return 2  # High Impact / Match Winner


# Set up Target Array (y) and Feature Matrix (X)
y_class = ml_dataset["next_match_runs"].apply(assign_performance_class)
X = ml_dataset[["rolling_avg_3", "balls_faced", "runs_volatility"]]

# Perform stratified train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_class, test_size=0.2, random_state=42, stratify=y_class
)

print(f"Train samples: {len(X_train)} | Test samples: {len(X_test)}")

Train samples: 37269 | Test samples: 9318


## 4. Training and Final Evaluation
The Random Forest Classifier acts like a panel of 150 mini-scouts (Decision Trees). It studies 80% of the historical data to find hidden patterns (e.g., "If a player has a high rolling average but massive volatility, they are highly likely to drop into Class 0 next game").

Finally, it tests itself on the remaining 20% of unseen data to generate a Performance Report, showing exactly how accurately it can spot upcoming failures, anchors, or match-winners before the players even step onto the field.

In [13]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

print("Training Balanced Random Forest Classifier matrix...")

# Hyperparameter-tuned and balanced architecture
clf_model = RandomForestClassifier(
    n_estimators=150,
    max_depth=8,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42,
)
clf_model.fit(X_train, y_train)

# Generate Predictions
y_pred = clf_model.predict(X_test)

# Report Outputs
print("\n" + "=" * 60)
print("        CRICKET CLASSIFICATION MODEL COMPILATION        ")
print("=" * 60)
print(f"Overall Subset Predictive Accuracy: {clf_model.score(X_test, y_test):.4f}\n")
print("--- Detailed Performance Classification Metrics ---")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=["Early Dismissal (<15)", "Anchor (15-40)", "Match Winner (>40)"],
        zero_division=0,
    )
)
print("=" * 60)

print("\n--- Updated Feature Contribution Weights ---")
for name, importance in zip(X.columns, clf_model.feature_importances_):
    print(f"Feature: {name:17} | Weight: {importance:.4f}")

Training Balanced Random Forest Classifier matrix...

        CRICKET CLASSIFICATION MODEL COMPILATION        
Overall Subset Predictive Accuracy: 0.4268

--- Detailed Performance Classification Metrics ---
                       precision    recall  f1-score   support

Early Dismissal (<15)       0.73      0.49      0.58      5634
       Anchor (15-40)       0.31      0.23      0.26      2636
   Match Winner (>40)       0.17      0.59      0.27      1048

             accuracy                           0.43      9318
            macro avg       0.40      0.44      0.37      9318
         weighted avg       0.55      0.43      0.46      9318


--- Updated Feature Contribution Weights ---
Feature: rolling_avg_3     | Weight: 0.4029
Feature: balls_faced       | Weight: 0.1658
Feature: runs_volatility   | Weight: 0.4313
